# Scientific Paper Domain Classification

**CSE440: Natural Language Processing II - Lab Project, Summer 2026**

**Group number:** Pending  
**Section:** Pending  
**Member 1:** Pending  
**Member 2:** Pending  
**Member 3:** Pending


# 1. Project Overview and Research Questions

This project receives the English abstract of a scientific paper and predicts one of seven parent domains: Computer Science, Electrical Engineering, Psychology, Mechanical Engineering, Civil Engineering, Medical Science, or Biochemistry.

The system may support literature organization, search, recommendation, reviewer routing, and indexing audits. A limitation is that interdisciplinary papers are forced into one parent label.

**RQ1:** Which TF-IDF traditional classifier performs best?  
**RQ2:** How do recurrent cell type and bidirectionality affect macro-F1?  
**RQ3:** Does BERT Base justify its additional computational cost?  
**RQ4:** Which domains are confused, and how can the errors be explained?


# 2. Kaggle Setup and Reproducibility

Attach the private `cse440-wos11967-assets` dataset before running. Select one stage per Kaggle session: `audit`, `traditional`, `rnn`, `bert`, or `final`.


In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import gc
import json
import random
import re
import shutil
import sys
import time
import unicodedata

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from wordcloud import STOPWORDS, WordCloud

print("Core libraries imported successfully.")


## Stage and Paths


In [ ]:
ACTIVE_STAGE = "audit"
VALID_STAGES = {"audit", "traditional", "rnn", "bert", "final"}
SEED = 42

INPUT_DIR = Path("/kaggle/input/cse440-wos11967-assets")
DATA_DIR = INPUT_DIR / "WOS-11967"
GLOVE_PATH = INPUT_DIR / "glove.6B.100d.txt"
ARTIFACT_INPUT_DIR = Path("/kaggle/input/cse440-project-artifacts")

WORK_DIR = Path("/kaggle/working/cse440-results")
FIGURES_DIR = WORK_DIR / "figures"
HISTORIES_DIR = WORK_DIR / "histories"
CHECKPOINTS_DIR = WORK_DIR / "checkpoints"
REPORTS_DIR = WORK_DIR / "reports"
PREDICTIONS_DIR = WORK_DIR / "predictions"

for directory in [WORK_DIR, FIGURES_DIR, HISTORIES_DIR, CHECKPOINTS_DIR, REPORTS_DIR, PREDICTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if ACTIVE_STAGE not in VALID_STAGES:
    raise ValueError(f"ACTIVE_STAGE must be one of {sorted(VALID_STAGES)}")

print("Active stage:", ACTIVE_STAGE)
print("Input directory:", INPUT_DIR)
print("Output directory:", WORK_DIR)


## Restore Previous Kaggle Outputs


In [ ]:
restored_files = 0

if ARTIFACT_INPUT_DIR.exists():
    nested_results = ARTIFACT_INPUT_DIR / "cse440-results"
    restore_source = nested_results if nested_results.exists() else ARTIFACT_INPUT_DIR

    for source_path in restore_source.rglob("*"):
        if source_path.is_file():
            relative_path = source_path.relative_to(restore_source)
            destination = WORK_DIR / relative_path
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.exists():
                shutil.copy2(source_path, destination)
                restored_files += 1

print("Restored artifact files:", restored_files)


## Random Seeds and Environment


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

package_names = ["numpy", "pandas", "scikit-learn", "tensorflow", "torch", "transformers"]
package_versions = {}
for package_name in package_names:
    try:
        package_versions[package_name] = version(package_name)
    except PackageNotFoundError:
        package_versions[package_name] = "not installed"

environment = pd.DataFrame({
    "Item": ["Python", "GPU available", "GPU"],
    "Value": [
        sys.version.split()[0],
        torch.cuda.is_available(),
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU runtime",
    ],
})

display(environment)
display(pd.DataFrame(package_versions.items(), columns=["Package", "Version"]))


# 3. Dataset Loading and Quality Checks

**Dataset:** Web of Science WOS-11967, version 6  
**Source:** https://data.mendeley.com/datasets/9rw3vkcfy4/6  
**DOI:** `10.17632/9rw3vkcfy4/6`  
**Licence:** CC BY 4.0

`X.txt` is the only model input and `YL1.txt` is the only target. `Y.txt`, `YL2.txt`, and metadata are excluded to prevent target leakage.


## Load Abstracts and Labels


In [ ]:
abstract_path = DATA_DIR / "X.txt"
label_path = DATA_DIR / "YL1.txt"

missing_files = [str(path) for path in [abstract_path, label_path] if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Attach the Kaggle input dataset. Missing: {missing_files}")

abstracts = abstract_path.read_text(encoding="utf-8-sig").splitlines()
raw_labels = label_path.read_text(encoding="utf-8-sig").splitlines()

print("Abstract rows:", len(abstracts))
print("Label rows:", len(raw_labels))


## Create the Modeling Dataset


In [ ]:
if len(abstracts) != len(raw_labels):
    raise ValueError("X.txt and YL1.txt are not aligned row-for-row.")

labels = pd.to_numeric(pd.Series(raw_labels), errors="coerce")
if labels.isna().any():
    raise ValueError("YL1.txt contains missing or malformed labels.")

label_names = {
    0: "Computer Science",
    1: "Electrical Engineering",
    2: "Psychology",
    3: "Mechanical Engineering",
    4: "Civil Engineering",
    5: "Medical Science",
    6: "Biochemistry",
}

labels = labels.astype(int)
unexpected_labels = sorted(set(labels) - set(label_names))
if unexpected_labels:
    raise ValueError(f"Unexpected labels: {unexpected_labels}")

papers = pd.DataFrame({
    "source_row": np.arange(len(abstracts)),
    "document_id": [f"wos11967_v6_{row:05d}" for row in range(len(abstracts))],
    "text": abstracts,
    "label": labels,
})
papers["domain"] = papers["label"].map(label_names)

display(papers.head())
print("Dataset shape:", papers.shape)


## Missing Values and Duplicates


In [ ]:
def normalize_text(text):
    normalized = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", normalized.lower()).strip()


papers["normalized_text"] = papers["text"].map(normalize_text)
papers["word_count"] = papers["text"].str.split().str.len()
papers["character_count"] = papers["text"].str.len()

exact_duplicate_rows = int(papers.duplicated("text", keep=False).sum())
normalized_duplicate_rows = int(papers.duplicated("normalized_text", keep=False).sum())
exact_conflicts = int((papers.groupby("text")["label"].nunique() > 1).sum())
normalized_conflicts = int((papers.groupby("normalized_text")["label"].nunique() > 1).sum())

quality_summary = pd.DataFrame({
    "Check": [
        "Missing abstracts",
        "Empty abstracts",
        "Missing labels",
        "Exact duplicate rows",
        "Normalized duplicate rows",
        "Exact duplicate groups with conflicting labels",
        "Normalized duplicate groups with conflicting labels",
    ],
    "Count": [
        int(papers["text"].isna().sum()),
        int(papers["normalized_text"].eq("").sum()),
        int(papers["label"].isna().sum()),
        exact_duplicate_rows,
        normalized_duplicate_rows,
        exact_conflicts,
        normalized_conflicts,
    ],
})

display(quality_summary)


## English-Language Evidence


In [ ]:
english_review_sample = pd.concat([
    class_rows.sample(n=min(3, len(class_rows)), random_state=SEED)
    for _, class_rows in papers.groupby("label", sort=True)
], ignore_index=True)

english_review_sample["excerpt"] = english_review_sample["text"].str.slice(0, 350)

display(english_review_sample[["document_id", "domain", "excerpt"]])
print("Manually review these", len(english_review_sample), "examples together with the dataset documentation.")


# 4. Exploratory Data Analysis

The following summaries describe the dataset. They do not fit a representation or use test performance.


## Class Distribution


In [ ]:
class_distribution = (
    papers.groupby(["label", "domain"], as_index=False)
    .size()
    .rename(columns={"size": "samples"})
)
class_distribution["percentage"] = class_distribution["samples"] / len(papers) * 100

display(class_distribution)

plt.figure(figsize=(10, 5))
sns.barplot(data=class_distribution, x="samples", y="domain", hue="domain", legend=False, palette="Set2")
plt.title("Class Distribution")
plt.xlabel("Number of Abstracts")
plt.ylabel("Domain")
plt.tight_layout()

class_figure_path = FIGURES_DIR / "class_distribution.png"
plt.savefig(class_figure_path, dpi=200, bbox_inches="tight")
plt.show()

imbalance_ratio = class_distribution["samples"].max() / class_distribution["samples"].min()
print("Largest-to-smallest class ratio:", round(imbalance_ratio, 2))


## Abstract Lengths


In [ ]:
length_summary = papers[["word_count", "character_count"]].describe(
    percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
).T

display(length_summary.round(2))
display(papers.groupby("domain")["word_count"].agg(["count", "mean", "median", "min", "max"]).round(2))

figure, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(papers["word_count"], bins=50, color="teal", ax=axes[0])
axes[0].set_title("Abstract Word Counts")
axes[0].set_xlabel("Number of Words")

sns.boxplot(data=papers, x="word_count", y="domain", hue="domain", legend=False, palette="Set2", ax=axes[1])
axes[1].set_title("Word Counts by Domain")
axes[1].set_ylabel("Domain")

plt.tight_layout()
length_figure_path = FIGURES_DIR / "text_length_analysis.png"
plt.savefig(length_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", length_figure_path)


## Word Clouds and Representative Examples


In [ ]:
figure, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for axis, (label, domain) in zip(axes, label_names.items()):
    domain_text = " ".join(papers.loc[papers["label"].eq(label), "text"])
    cloud = WordCloud(
        width=900,
        height=450,
        background_color="white",
        stopwords=STOPWORDS,
        max_words=100,
        random_state=SEED,
        collocations=False,
    ).generate(domain_text)
    axis.imshow(cloud, interpolation="bilinear")
    axis.set_title(domain)
    axis.axis("off")

axes[-1].axis("off")
plt.tight_layout()
wordcloud_path = FIGURES_DIR / "class_wordclouds.png"
plt.savefig(wordcloud_path, dpi=180, bbox_inches="tight")
plt.show()

representative_examples = pd.concat([
    class_rows.sample(n=min(2, len(class_rows)), random_state=SEED)
    for _, class_rows in papers.groupby("label", sort=True)
], ignore_index=True)
representative_examples["excerpt"] = representative_examples["text"].str.slice(0, 350)

display(representative_examples[["document_id", "domain", "word_count", "excerpt"]])
print("Saved:", wordcloud_path)


# 5. Preprocessing and Frozen Split

TF-IDF uses lowercase text with safe whitespace normalization. GloVe uses a training-fitted Keras tokenizer. BERT uses its official tokenizer with minimal normalization. No representation is fitted on validation or test text.


## Create or Reload the Split Manifest


In [ ]:
manifest_path = WORK_DIR / "split_manifest.csv"

if manifest_path.exists():
    split_manifest = pd.read_csv(manifest_path)
    if set(split_manifest["document_id"]) != set(papers["document_id"]):
        raise ValueError("The saved split manifest does not match the attached dataset.")
    split_method = "Reloaded saved manifest"
else:
    papers["duplicate_group"] = pd.factorize(papers["normalized_text"])[0]

    if normalized_duplicate_rows > 0:
        splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
        fold_numbers = np.full(len(papers), -1, dtype=int)

        for fold, (_, held_out_rows) in enumerate(
            splitter.split(papers, y=papers["label"], groups=papers["duplicate_group"])
        ):
            fold_numbers[held_out_rows] = fold

        split_names = np.where(
            fold_numbers < 14,
            "train",
            np.where(fold_numbers < 17, "validation", "test"),
        )
        split_method = "StratifiedGroupKFold"
    else:
        train_rows, remaining_rows = train_test_split(
            papers.index,
            test_size=0.30,
            random_state=SEED,
            stratify=papers["label"],
        )
        validation_rows, test_rows = train_test_split(
            remaining_rows,
            test_size=0.50,
            random_state=SEED,
            stratify=papers.loc[remaining_rows, "label"],
        )

        split_names = np.full(len(papers), "", dtype=object)
        split_names[train_rows] = "train"
        split_names[validation_rows] = "validation"
        split_names[test_rows] = "test"
        split_method = "Two-stage stratified split"

    papers["split"] = split_names
    split_manifest = papers[["document_id", "source_row", "label", "domain", "duplicate_group", "split"]].copy()
    split_manifest.to_csv(manifest_path, index=False)

display(split_manifest.groupby(["domain", "split"]).size().unstack(fill_value=0))
print("Split method:", split_method)
print("Saved manifest:", manifest_path)


## Validate and Apply the Split


In [ ]:
if exact_conflicts or normalized_conflicts:
    raise ValueError("Conflicting duplicate labels must be reviewed before training.")

if split_manifest["document_id"].duplicated().any():
    raise ValueError("A document appears more than once in the split manifest.")

if split_manifest.groupby("duplicate_group")["split"].nunique().max() > 1:
    raise ValueError("A normalized duplicate group crosses data splits.")

if not split_manifest.groupby("split")["label"].nunique().eq(7).all():
    raise ValueError("Every split must contain all seven classes.")

papers = papers.drop(columns=["split"], errors="ignore").merge(
    split_manifest[["document_id", "split"]],
    on="document_id",
    how="left",
    validate="one_to_one",
)

train_data = papers.loc[papers["split"].eq("train")].reset_index(drop=True)
validation_data = papers.loc[papers["split"].eq("validation")].reset_index(drop=True)
test_data = papers.loc[papers["split"].eq("test")].reset_index(drop=True)

split_sizes = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Samples": [len(train_data), len(validation_data), len(test_data)],
})

display(split_sizes)
print("Split validation passed. Test labels remain unused until ACTIVE_STAGE is final.")


## Experiment Results Table


In [ ]:
tuning_path = WORK_DIR / "tuning_results.csv"
tuning_columns = [
    "model",
    "family",
    "representation",
    "config_id",
    "hyperparameters",
    "validation_accuracy",
    "validation_macro_f1",
    "best_epoch",
    "training_time_seconds",
    "checkpoint_path",
]

if tuning_path.exists():
    tuning_results = pd.read_csv(tuning_path).reindex(columns=tuning_columns)
else:
    tuning_results = pd.DataFrame(columns=tuning_columns)

def record_tuning_result(result):
    global tuning_results
    existing_result = (
        tuning_results["model"].eq(result["model"])
        & tuning_results["config_id"].eq(result["config_id"])
    )
    tuning_results = tuning_results.loc[~existing_result].copy()
    tuning_results = pd.concat([tuning_results, pd.DataFrame([result])], ignore_index=True)
    tuning_results.to_csv(tuning_path, index=False)


def validation_scores(true_labels, predictions):
    return {
        "validation_accuracy": accuracy_score(true_labels, predictions),
        "validation_macro_f1": f1_score(true_labels, predictions, average="macro"),
    }


print("Existing official tuning runs:", len(tuning_results))
print("Tuning table:", tuning_path)


## Leakage Assertions

The assertions below must pass before any training stage. The test set is evaluated only when `ACTIVE_STAGE = "final"` and all 30 validation runs are available.


In [ ]:
assert len(papers) >= 10000
assert papers["label"].nunique() == 7
assert papers["normalized_text"].ne("").all()
assert set(train_data["document_id"]).isdisjoint(validation_data["document_id"])
assert set(train_data["document_id"]).isdisjoint(test_data["document_id"])
assert set(validation_data["document_id"]).isdisjoint(test_data["document_id"])

print("Pre-training assertions passed.")


# 6. TF-IDF and Traditional Models

Fit TF-IDF on training text only. Each of the three required classifiers uses three manually defined configurations.


## TF-IDF Representation


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = None
x_train_tfidf = None
x_validation_tfidf = None

if ACTIVE_STAGE in {"traditional", "final"}:
    tfidf_vectorizer = TfidfVectorizer(
        lowercase=True,
        preprocessor=normalize_text,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=50000,
        sublinear_tf=True,
    )
    x_train_tfidf = tfidf_vectorizer.fit_transform(train_data["text"])
    x_validation_tfidf = tfidf_vectorizer.transform(validation_data["text"])
    joblib.dump(tfidf_vectorizer, WORK_DIR / "tfidf_vectorizer.joblib")
    print("Training TF-IDF shape:", x_train_tfidf.shape)
    print("Validation TF-IDF shape:", x_validation_tfidf.shape)
else:
    print("TF-IDF fitting is reserved for the traditional or final stage.")


## Manual Traditional Configurations


In [ ]:
traditional_configurations = [
    {"model": "Logistic Regression", "config_id": "LR_1", "parameters": {"C": 0.5}},
    {"model": "Logistic Regression", "config_id": "LR_2", "parameters": {"C": 1.0}},
    {"model": "Logistic Regression", "config_id": "LR_3", "parameters": {"C": 2.0}},
    {"model": "Multinomial Naive Bayes", "config_id": "NB_1", "parameters": {"alpha": 0.1}},
    {"model": "Multinomial Naive Bayes", "config_id": "NB_2", "parameters": {"alpha": 0.5}},
    {"model": "Multinomial Naive Bayes", "config_id": "NB_3", "parameters": {"alpha": 1.0}},
    {"model": "Random Forest", "config_id": "RF_1", "parameters": {"n_estimators": 200, "max_depth": 50}},
    {"model": "Random Forest", "config_id": "RF_2", "parameters": {"n_estimators": 300, "max_depth": 100}},
    {"model": "Random Forest", "config_id": "RF_3", "parameters": {"n_estimators": 400, "max_depth": None}},
]

display(pd.DataFrame(traditional_configurations))


## Train and Validate Traditional Models


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

def create_traditional_model(model_name, parameters):
    if model_name == "Logistic Regression":
        return LogisticRegression(max_iter=1000, random_state=SEED, **parameters)
    if model_name == "Multinomial Naive Bayes":
        return MultinomialNB(**parameters)
    if model_name == "Random Forest":
        return RandomForestClassifier(random_state=SEED, n_jobs=-1, **parameters)
    raise ValueError(f"Unknown traditional model: {model_name}")


if ACTIVE_STAGE == "traditional":
    completed_runs = set(zip(tuning_results["model"], tuning_results["config_id"]))

    for configuration in traditional_configurations:
        run_id = configuration["config_id"]
        run_key = (configuration["model"], configuration["config_id"])
        if run_key in completed_runs:
            print("Skipping completed run:", run_id)
            continue

        model = create_traditional_model(configuration["model"], configuration["parameters"])
        start_time = time.perf_counter()
        model.fit(x_train_tfidf, train_data["label"])
        validation_predictions = model.predict(x_validation_tfidf)
        elapsed = time.perf_counter() - start_time

        checkpoint = CHECKPOINTS_DIR / f"{run_id}.joblib"
        joblib.dump(model, checkpoint)
        scores = validation_scores(validation_data["label"], validation_predictions)

        record_tuning_result({
            "model": configuration["model"],
            "family": "Traditional",
            "representation": "TF-IDF",
            "config_id": configuration["config_id"],
            "hyperparameters": json.dumps(configuration["parameters"], sort_keys=True),
            **scores,
            "best_epoch": np.nan,
            "training_time_seconds": elapsed,
            "checkpoint_path": str(checkpoint.relative_to(WORK_DIR)),
        })
        print(run_id, scores)

    display(tuning_results.loc[tuning_results["family"].eq("Traditional")].sort_values("validation_macro_f1", ascending=False))
else:
    print("Set ACTIVE_STAGE to traditional to run the nine traditional experiments.")


# 7. GloVe and Sequence Preparation

The Keras tokenizer is fitted on training text only. The vocabulary is capped at 30,000 words and sequence length is the training 95th percentile capped at 400.


In [ ]:
NUM_WORDS = 30000
EMBEDDING_DIM = 100

tokenizer = None
embedding_matrix = None
x_train_sequences = None
x_validation_sequences = None
x_test_sequences = None
sequence_length = None

if ACTIVE_STAGE in {"rnn", "final"}:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.preprocessing.text import Tokenizer

    tf.keras.utils.set_random_seed(SEED)

    tokenizer = Tokenizer(num_words=NUM_WORDS, oov_token="<OOV>", lower=True)
    tokenizer.fit_on_texts(train_data["text"])

    training_lengths = train_data["text"].str.split().str.len()
    sequence_length = min(400, int(np.ceil(training_lengths.quantile(0.95))))

    x_train_sequences = pad_sequences(
        tokenizer.texts_to_sequences(train_data["text"]),
        maxlen=sequence_length,
        padding="post",
        truncating="post",
    )
    x_validation_sequences = pad_sequences(
        tokenizer.texts_to_sequences(validation_data["text"]),
        maxlen=sequence_length,
        padding="post",
        truncating="post",
    )
    if ACTIVE_STAGE == "final":
        x_test_sequences = pad_sequences(
            tokenizer.texts_to_sequences(test_data["text"]),
            maxlen=sequence_length,
            padding="post",
            truncating="post",
        )

    print("Vocabulary entries:", min(NUM_WORDS, len(tokenizer.word_index) + 1))
    print("Sequence length:", sequence_length)
    print("Training sequence shape:", x_train_sequences.shape)
else:
    print("GloVe preparation is reserved for the rnn or final stage.")


## Load GloVe and Measure Coverage


In [ ]:
if ACTIVE_STAGE in {"rnn", "final"}:
    if not GLOVE_PATH.is_file():
        raise FileNotFoundError(f"Missing GloVe input: {GLOVE_PATH}")

    vocabulary_size = min(NUM_WORDS, len(tokenizer.word_index) + 1)
    needed_words = {
        word for word, index in tokenizer.word_index.items()
        if index < vocabulary_size
    }
    glove_vectors = {}

    with GLOVE_PATH.open("r", encoding="utf-8") as glove_file:
        for line in glove_file:
            values = line.rstrip().split()
            word = values[0]
            if word in needed_words:
                glove_vectors[word] = np.asarray(values[1:], dtype="float32")

    rng = np.random.default_rng(SEED)
    embedding_matrix = rng.normal(0, 0.05, size=(vocabulary_size, EMBEDDING_DIM)).astype("float32")
    embedding_matrix[0] = 0

    covered_words = 0
    for word, index in tokenizer.word_index.items():
        if index >= vocabulary_size:
            continue
        vector = glove_vectors.get(word)
        if vector is not None:
            embedding_matrix[index] = vector
            covered_words += 1

    vocabulary_coverage = covered_words / max(1, vocabulary_size - 2)
    validation_token_count = np.count_nonzero(x_validation_sequences)
    validation_oov_count = np.count_nonzero(x_validation_sequences == tokenizer.word_index["<OOV>"])
    validation_oov_rate = validation_oov_count / max(1, validation_token_count)

    coverage_table = pd.DataFrame({
        "Measure": ["GloVe vocabulary coverage", "Validation token OOV rate"],
        "Value": [vocabulary_coverage, validation_oov_rate],
    })
    display(coverage_table)
else:
    print("GloVe vectors were not loaded in this stage.")


# 8. Recurrent Neural Models

TensorFlow/Keras is used for SimpleRNN, GRU, LSTM, and their bidirectional variants. All models share the same tokenizer, GloVe matrix, batch size, epoch limit, and early-stopping rule.


## Recurrent Configurations


In [ ]:
recurrent_profiles = {
    "A": {"units": 64, "trainable_glove": False, "dropout": 0.3, "learning_rate": 1e-3},
    "B": {"units": 128, "trainable_glove": False, "dropout": 0.3, "learning_rate": 1e-3},
    "C": {"units": 128, "trainable_glove": True, "dropout": 0.5, "learning_rate": 5e-4},
}
recurrent_models = [
    "SimpleRNN",
    "GRU",
    "LSTM",
    "Bidirectional SimpleRNN",
    "Bidirectional GRU",
    "Bidirectional LSTM",
]

recurrent_configuration_table = pd.DataFrame([
    {"Model": model_name, "Profile": profile_name, **profile}
    for model_name in recurrent_models
    for profile_name, profile in recurrent_profiles.items()
])
display(recurrent_configuration_table)


## Recurrent Model Builder


In [ ]:
def build_recurrent_model(model_name, profile):
    import tensorflow as tf
    from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Embedding, GRU, LSTM, SimpleRNN
    from tensorflow.keras.models import Sequential

    layer_types = {
        "SimpleRNN": SimpleRNN,
        "GRU": GRU,
        "LSTM": LSTM,
    }
    base_name = model_name.replace("Bidirectional ", "")
    recurrent_layer = layer_types[base_name](profile["units"])
    if model_name.startswith("Bidirectional"):
        recurrent_layer = Bidirectional(recurrent_layer)

    model = Sequential([
        Embedding(
            input_dim=embedding_matrix.shape[0],
            output_dim=EMBEDDING_DIM,
            weights=[embedding_matrix],
            trainable=profile["trainable_glove"],
            mask_zero=True,
        ),
        recurrent_layer,
        Dropout(profile["dropout"]),
        Dense(7, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=profile["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


print("Recurrent model builder is ready.")


## Train and Validate Recurrent Models


In [ ]:
if ACTIVE_STAGE == "rnn":
    import tensorflow as tf

    completed_runs = set(zip(tuning_results["model"], tuning_results["config_id"]))

    for model_name in recurrent_models:
        for profile_name, profile in recurrent_profiles.items():
            run_id = f"RNN_{model_name.replace(' ', '_')}_{profile_name}"
            run_key = (model_name, profile_name)
            if run_key in completed_runs:
                print("Skipping completed run:", run_id)
                continue

            tf.keras.backend.clear_session()
            model = build_recurrent_model(model_name, profile)
            checkpoint = CHECKPOINTS_DIR / f"{run_id}.weights.h5"

            callbacks = [
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=2,
                    restore_best_weights=True,
                ),
                tf.keras.callbacks.ModelCheckpoint(
                    str(checkpoint),
                    monitor="val_loss",
                    save_best_only=True,
                    save_weights_only=True,
                ),
            ]

            start_time = time.perf_counter()
            history = model.fit(
                x_train_sequences,
                train_data["label"].to_numpy(),
                validation_data=(x_validation_sequences, validation_data["label"].to_numpy()),
                epochs=15,
                batch_size=64,
                callbacks=callbacks,
                verbose=1,
            )
            elapsed = time.perf_counter() - start_time

            validation_predictions = model.predict(x_validation_sequences, batch_size=64, verbose=0).argmax(axis=1)
            scores = validation_scores(validation_data["label"], validation_predictions)
            best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

            history_path = HISTORIES_DIR / f"{run_id}.csv"
            pd.DataFrame(history.history).to_csv(history_path, index=False)

            parameters = {"model_name": model_name, "profile": profile_name, **profile}
            record_tuning_result({
                "model": model_name,
                "family": "Recurrent",
                "representation": "GloVe 6B 100d",
                "config_id": profile_name,
                "hyperparameters": json.dumps(parameters, sort_keys=True),
                **scores,
                "best_epoch": best_epoch,
                "training_time_seconds": elapsed,
                "checkpoint_path": str(checkpoint.relative_to(WORK_DIR)),
            })
            print(run_id, scores)

            del model
            gc.collect()

    display(tuning_results.loc[tuning_results["family"].eq("Recurrent")].sort_values("validation_macro_f1", ascending=False))
else:
    print("Set ACTIVE_STAGE to rnn to run the 18 recurrent experiments.")


# 9. BERT Base

BERT uses the official `bert-base-uncased` tokenizer. Maximum length is selected from training WordPiece lengths only: 256 when no more than 5% exceed 256 tokens, otherwise 384.


## BERT Tokenization


In [ ]:
BERT_MODEL_SOURCE = "bert-base-uncased"
bert_tokenizer = None
bert_train_dataset = None
bert_validation_dataset = None
bert_test_dataset = None
bert_max_length = None

if ACTIVE_STAGE in {"bert", "final"}:
    from transformers import AutoTokenizer

    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_SOURCE)
    training_token_ids = bert_tokenizer(
        train_data["text"].tolist(),
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]
    training_token_lengths = np.array([len(token_ids) for token_ids in training_token_ids])
    percentage_over_256 = float(np.mean(training_token_lengths > 256) * 100)
    bert_max_length = 256 if percentage_over_256 <= 5 else 384

    token_length_summary = pd.DataFrame({
        "Measure": ["Median", "95th percentile", "Maximum", "Percent over 256", "Selected maximum length"],
        "Value": [
            np.median(training_token_lengths),
            np.percentile(training_token_lengths, 95),
            training_token_lengths.max(),
            percentage_over_256,
            bert_max_length,
        ],
    })
    display(token_length_summary)
else:
    print("BERT tokenization is reserved for the bert or final stage.")


## BERT Datasets and Metrics


In [ ]:
class BertAbstractDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: value[index] for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item


def make_bert_dataset(texts, labels):
    encodings = bert_tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=bert_max_length,
        return_tensors="pt",
    )
    return BertAbstractDataset(encodings, np.asarray(labels, dtype=int))


def bert_metrics(prediction_output):
    predictions = np.argmax(prediction_output.predictions, axis=1)
    labels = prediction_output.label_ids
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }


if ACTIVE_STAGE in {"bert", "final"}:
    bert_train_dataset = make_bert_dataset(train_data["text"], train_data["label"])
    bert_validation_dataset = make_bert_dataset(validation_data["text"], validation_data["label"])
    if ACTIVE_STAGE == "final":
        bert_test_dataset = make_bert_dataset(test_data["text"], test_data["label"])
    print("BERT training samples:", len(bert_train_dataset))
    print("BERT validation samples:", len(bert_validation_dataset))
else:
    print("BERT datasets were not created in this stage.")


## Manual BERT Configurations


In [ ]:
bert_configurations = [
    {"config_id": "BERT_1", "learning_rate": 1e-5},
    {"config_id": "BERT_2", "learning_rate": 2e-5},
    {"config_id": "BERT_3", "learning_rate": 3e-5},
]

display(pd.DataFrame(bert_configurations))


## Train and Validate BERT


In [ ]:
if ACTIVE_STAGE == "bert":
    from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

    completed_runs = set(zip(tuning_results["model"], tuning_results["config_id"]))

    for configuration in bert_configurations:
        run_id = configuration["config_id"]
        run_key = ("BERT Base", configuration["config_id"])
        if run_key in completed_runs:
            print("Skipping completed run:", run_id)
            continue

        output_directory = CHECKPOINTS_DIR / run_id
        model = AutoModelForSequenceClassification.from_pretrained(
            BERT_MODEL_SOURCE,
            num_labels=7,
        )
        arguments = TrainingArguments(
            output_dir=str(output_directory),
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="epoch",
            learning_rate=configuration["learning_rate"],
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            num_train_epochs=3,
            weight_decay=0.01,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            save_total_limit=1,
            fp16=torch.cuda.is_available(),
            report_to="none",
            seed=SEED,
        )
        trainer = Trainer(
            model=model,
            args=arguments,
            train_dataset=bert_train_dataset,
            eval_dataset=bert_validation_dataset,
            processing_class=bert_tokenizer,
            compute_metrics=bert_metrics,
        )

        start_time = time.perf_counter()
        trainer.train()
        elapsed = time.perf_counter() - start_time
        validation_output = trainer.predict(bert_validation_dataset)
        validation_predictions = validation_output.predictions.argmax(axis=1)
        scores = validation_scores(validation_data["label"], validation_predictions)

        evaluation_logs = [
            entry for entry in trainer.state.log_history
            if "eval_macro_f1" in entry
        ]
        best_log = max(evaluation_logs, key=lambda entry: entry["eval_macro_f1"])
        best_epoch = int(round(best_log["epoch"]))

        best_model_directory = output_directory / "best_model"
        trainer.save_model(best_model_directory)
        bert_tokenizer.save_pretrained(best_model_directory)

        parameters = {
            "learning_rate": configuration["learning_rate"],
            "batch_size": 8,
            "epochs": 3,
            "weight_decay": 0.01,
            "max_length": bert_max_length,
        }
        record_tuning_result({
            "model": "BERT Base",
            "family": "Transformer",
            "representation": "BERT WordPiece",
            "config_id": configuration["config_id"],
            "hyperparameters": json.dumps(parameters, sort_keys=True),
            **scores,
            "best_epoch": best_epoch,
            "training_time_seconds": elapsed,
            "checkpoint_path": str(best_model_directory.relative_to(WORK_DIR)),
        })
        print(run_id, scores)

        del trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    display(tuning_results.loc[tuning_results["family"].eq("Transformer")].sort_values("validation_macro_f1", ascending=False))
else:
    print("Set ACTIVE_STAGE to bert to run the three BERT experiments.")


# 10. Validation Results and Selection

The best configuration of each required model is selected by validation macro-F1. Validation accuracy breaks an exact tie.


## Verify the 30 Official Runs


In [ ]:
required_models = [
    "Logistic Regression",
    "Multinomial Naive Bayes",
    "Random Forest",
    "SimpleRNN",
    "GRU",
    "LSTM",
    "Bidirectional SimpleRNN",
    "Bidirectional GRU",
    "Bidirectional LSTM",
    "BERT Base",
]

run_counts = (
    tuning_results.groupby("model")
    .size()
    .reindex(required_models, fill_value=0)
    .rename("Completed configurations")
    .reset_index(names="Model")
)
display(run_counts)
print("Total completed official runs:", len(tuning_results))


## Lock the Best Configuration per Model


In [ ]:
selected_path = WORK_DIR / "selected_configurations.csv"
selected_configurations = pd.DataFrame()

if ACTIVE_STAGE == "final":
    if len(tuning_results) != 30 or not run_counts["Completed configurations"].eq(3).all():
        raise RuntimeError("Complete exactly three validation runs for every required model before final testing.")

    selected_configurations = (
        tuning_results.sort_values(
            ["model", "validation_macro_f1", "validation_accuracy"],
            ascending=[True, False, False],
        )
        .groupby("model", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    selected_configurations.to_csv(selected_path, index=False)
    display(selected_configurations[[
        "model", "representation", "config_id",
        "validation_accuracy", "validation_macro_f1",
    ]])
    print("Locked configurations:", selected_path)
else:
    print("Configuration locking occurs only in the final stage.")


# 11. Final Test Evaluation and Error Analysis

This section is guarded by `ACTIVE_STAGE = "final"` and a complete validation table. If `final_test_results.csv` already exists, test evaluation is skipped to prevent repeated checking.


## Evaluation Helper


In [ ]:
final_results_path = WORK_DIR / "final_test_results.csv"

def model_slug(model_name):
    return re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")


def save_test_evaluation(model_name, representation, true_labels, predictions, selected_row):
    label_order = list(label_names)
    target_names = [label_names[label] for label in label_order]
    report = classification_report(
        true_labels,
        predictions,
        labels=label_order,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )
    report_frame = pd.DataFrame(report).T
    slug = model_slug(model_name)
    report_frame.to_csv(REPORTS_DIR / f"{slug}_classification_report.csv")

    raw_matrix = confusion_matrix(true_labels, predictions, labels=label_order)
    normalized_matrix = confusion_matrix(true_labels, predictions, labels=label_order, normalize="true")

    figure, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(raw_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names, ax=axes[0])
    axes[0].set_title(f"{model_name}: Raw Confusion Matrix")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")

    sns.heatmap(normalized_matrix, annot=True, fmt=".2f", cmap="Greens", xticklabels=target_names, yticklabels=target_names, ax=axes[1])
    axes[1].set_title(f"{model_name}: Row-Normalized Matrix")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{slug}_confusion_matrices.png", dpi=180, bbox_inches="tight")
    plt.show()

    prediction_frame = pd.DataFrame({
        "document_id": test_data["document_id"],
        "true_label": np.asarray(true_labels),
        "predicted_label": np.asarray(predictions),
    })
    prediction_frame.to_csv(PREDICTIONS_DIR / f"{slug}.csv", index=False)

    display(report_frame)
    return {
        "model": model_name,
        "representation": representation,
        "config_id": selected_row["config_id"],
        "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, average="macro"),
        "weighted_f1": f1_score(true_labels, predictions, average="weighted"),
        "training_time_seconds": selected_row["training_time_seconds"],
    }


print("Final evaluation helper is ready.")


## Evaluate the Ten Locked Models Once


In [ ]:
if final_results_path.exists():
    final_test_results = pd.read_csv(final_results_path)
else:
    final_test_results = pd.DataFrame()

def record_final_result(result):
    global final_test_results
    if not final_test_results.empty:
        final_test_results = final_test_results.loc[
            final_test_results["model"].ne(result["model"])
        ].copy()
    final_test_results = pd.concat(
        [final_test_results, pd.DataFrame([result])],
        ignore_index=True,
    ).sort_values("macro_f1", ascending=False).reset_index(drop=True)
    final_test_results.to_csv(final_results_path, index=False)


if ACTIVE_STAGE == "final":
    completed_test_models = set(final_test_results.get("model", pd.Series(dtype=str)))

    x_test_tfidf = tfidf_vectorizer.transform(test_data["text"])
    for model_name in ["Logistic Regression", "Multinomial Naive Bayes", "Random Forest"]:
        if model_name in completed_test_models:
            print("Skipping completed test evaluation:", model_name)
            continue
        selected_row = selected_configurations.loc[selected_configurations["model"].eq(model_name)].iloc[0]
        checkpoint = WORK_DIR / selected_row["checkpoint_path"]
        model = joblib.load(checkpoint)
        predictions = model.predict(x_test_tfidf)
        result = save_test_evaluation(model_name, "TF-IDF", test_data["label"], predictions, selected_row)
        record_final_result(result)
        completed_test_models.add(model_name)

    import tensorflow as tf
    for model_name in recurrent_models:
        if model_name in completed_test_models:
            print("Skipping completed test evaluation:", model_name)
            continue
        selected_row = selected_configurations.loc[selected_configurations["model"].eq(model_name)].iloc[0]
        parameters = json.loads(selected_row["hyperparameters"])
        profile = {
            "units": parameters["units"],
            "trainable_glove": parameters["trainable_glove"],
            "dropout": parameters["dropout"],
            "learning_rate": parameters["learning_rate"],
        }
        tf.keras.backend.clear_session()
        model = build_recurrent_model(model_name, profile)
        model.build((None, sequence_length))
        checkpoint = WORK_DIR / selected_row["checkpoint_path"]
        model.load_weights(checkpoint)
        predictions = model.predict(x_test_sequences, batch_size=64, verbose=0).argmax(axis=1)
        result = save_test_evaluation(model_name, "GloVe 6B 100d", test_data["label"], predictions, selected_row)
        record_final_result(result)
        completed_test_models.add(model_name)
        del model

    tf.keras.backend.clear_session()
    gc.collect()

    from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

    if "BERT Base" not in completed_test_models:
        selected_row = selected_configurations.loc[selected_configurations["model"].eq("BERT Base")].iloc[0]
        bert_checkpoint = WORK_DIR / selected_row["checkpoint_path"]
        bert_model = AutoModelForSequenceClassification.from_pretrained(bert_checkpoint)
        prediction_arguments = TrainingArguments(
            output_dir=str(WORK_DIR / "bert_final_prediction"),
            per_device_eval_batch_size=8,
            fp16=torch.cuda.is_available(),
            report_to="none",
        )
        bert_trainer = Trainer(model=bert_model, args=prediction_arguments)
        bert_output = bert_trainer.predict(bert_test_dataset)
        bert_predictions = bert_output.predictions.argmax(axis=1)
        result = save_test_evaluation(
            "BERT Base", "BERT WordPiece", test_data["label"], bert_predictions, selected_row
        )
        record_final_result(result)

    if set(final_test_results["model"]) != set(required_models):
        raise RuntimeError("Final evaluation is incomplete; resume the final stage without changing selections.")

    display(final_test_results)
    print("Final test results saved:", final_results_path)
else:
    print("Final test evaluation is locked until ACTIVE_STAGE is final.")


## Model Comparison


In [ ]:
if not final_test_results.empty:
    display(final_test_results)

    plt.figure(figsize=(11, 6))
    sns.barplot(data=final_test_results, x="macro_f1", y="model", hue="representation", dodge=False)
    plt.title("Final Test Macro-F1 by Model")
    plt.xlabel("Macro-F1")
    plt.ylabel("Model")
    plt.xlim(0, 1)
    plt.tight_layout()
    comparison_path = FIGURES_DIR / "final_model_comparison.png"
    plt.savefig(comparison_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Best model:", final_test_results.iloc[0]["model"])
    print("Worst model:", final_test_results.iloc[-1]["model"])
else:
    print("Run the final stage before creating the model comparison.")


## Error Analysis


In [ ]:
confusion_pairs = pd.DataFrame()

if not final_test_results.empty:
    best_model_name = final_test_results.iloc[0]["model"]
    best_predictions = pd.read_csv(PREDICTIONS_DIR / f"{model_slug(best_model_name)}.csv")
    errors = best_predictions.loc[best_predictions["true_label"].ne(best_predictions["predicted_label"])].copy()
    confusion_pairs = (
        errors.groupby(["true_label", "predicted_label"])
        .size()
        .reset_index(name="errors")
        .sort_values("errors", ascending=False)
    )
    confusion_pairs["true_domain"] = confusion_pairs["true_label"].map(label_names)
    confusion_pairs["predicted_domain"] = confusion_pairs["predicted_label"].map(label_names)

    display(confusion_pairs[["true_domain", "predicted_domain", "errors"]].head(10))

    error_examples = errors.merge(test_data[["document_id", "text"]], on="document_id", how="left")
    error_examples["true_domain"] = error_examples["true_label"].map(label_names)
    error_examples["predicted_domain"] = error_examples["predicted_label"].map(label_names)
    error_examples["excerpt"] = error_examples["text"].str.slice(0, 450)
    display(error_examples[["document_id", "true_domain", "predicted_domain", "excerpt"]].head(15))
else:
    print("Error analysis becomes available after final test evaluation.")


# 12. Research Questions, Limitations, and Conclusion

The following cell extracts the numerical evidence for the four research questions. Use the displayed evidence to write the final interpretation in your own words after execution.


In [ ]:
if not final_test_results.empty:
    traditional_results = final_test_results.loc[final_test_results["model"].isin([
        "Logistic Regression", "Multinomial Naive Bayes", "Random Forest"
    ])]
    recurrent_results = final_test_results.loc[final_test_results["model"].isin(recurrent_models)]
    bert_result = final_test_results.loc[final_test_results["model"].eq("BERT Base")].iloc[0]

    rq_evidence = pd.DataFrame({
        "Research question": ["RQ1", "RQ2", "RQ3", "RQ4"],
        "Evidence": [
            f"Best traditional model: {traditional_results.iloc[0]['model']} with macro-F1 {traditional_results.iloc[0]['macro_f1']:.4f}",
            f"Best recurrent model: {recurrent_results.iloc[0]['model']} with macro-F1 {recurrent_results.iloc[0]['macro_f1']:.4f}",
            f"BERT macro-F1 {bert_result['macro_f1']:.4f}; best non-BERT macro-F1 {final_test_results.loc[final_test_results['model'].ne('BERT Base'), 'macro_f1'].max():.4f}",
            "Most common confusion: " + (
                f"{confusion_pairs.iloc[0]['true_domain']} to {confusion_pairs.iloc[0]['predicted_domain']}"
                if not confusion_pairs.empty else "No errors"
            ),
        ],
    })
    display(rq_evidence)
else:
    print("Research-question evidence becomes available after final test evaluation.")


## Limitations and Final Interpretation

Discuss the following using the executed results:

- One parent label cannot fully represent interdisciplinary papers.
- Class frequencies differ, so macro-F1 is the primary selection metric.
- GloVe may miss specialized scientific vocabulary.
- RNN truncation and BERT WordPiece truncation may remove information from long abstracts.
- Computational comparisons depend on the Kaggle accelerator and session conditions.

Add the final answers to RQ1-RQ4, key conclusions, and realistic future improvements only after all displayed evidence has been generated.


## Reproducibility Summary

Before submission, Quick Save the fully executed Kaggle notebook, download the `.ipynb`, and confirm that every code cell has visible output. Submit one notebook named according to the instructor-approved three-member filename format.
